# Multi-Class Workout Recommendation Research Project
## Complete Clean Jupyter Notebook

# 1. Install Required Libraries

In [ ]:
!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn scipy imbalanced-learn joblib


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier


# 3. Load Dataset

In [ ]:
DATA_PATH = 'scraped_style_noisy_workout_dataset (1).csv'

df_raw = pd.read_csv(DATA_PATH)

print(df_raw.shape)

df_raw.head()

(10800, 34)


,row_id,timestamp,user_age,user_bmi,user_body_fat_pct,fitness_level,experience_years,resting_heart_rate,max_heart_rate,heart_rate_reserve,...,motivation_score,workout_name,muscle_groups,equipment_needed,target_workout_type,target_duration_min,target_intensity_level,estimated_calories,data_source,sleep_hours
0,4280,2026-05-02T00:16:17.000362,23.0,NaN,18.6,1,1,64.0,197,133,...,6.0,Bro Split - Advanced Day 5,"shoulders,chest,back",True,strength,31.0,9,83.0,bodybuilding,NaN
1,5086,2025-10-24T00:16:17.022219,47.0,27.1,19.9,1,0,67.0,173,106,...,6.0,Bro Split - Novice Day 3,"core,back,legs",True,strength,21.0,5,56.0,bodybuilding,NaN
2,6481,2026-03-06T00:16:17.064623,23.0,25.7,20.7,1,1,52.0,197,145,...,NaN,WOD-2719,full_body,True,hiit,10.0,6,47.0,crossfit,NaN
3,6830,2025-07-13T00:16:17.075315,30.0,28.3,22.2,10,7,63.0,190,127,...,10.0,Bro Split - Intermediate Day 5,"arms,legs,chest",True,strength,72.0,8,203.0,bodybuilding,NaN
4,1059,2025-09-27T00:16:16.902704,47.0,25.3,25.2,1,1,62.0,173,111,...,2.0,WOD-8525,full_body,True,hiit,14.0,7,64.0,crossfit,NaN


# 4. Dataset Overview

In [ ]:
print(df_raw.info())

print(df_raw.isnull().sum())

df_raw.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   row_id                   10800 non-null  int64  
 1   timestamp                10800 non-null  object 
 2   user_age                 9181 non-null   float64
 3   user_bmi                 9463 non-null   float64
 4   user_body_fat_pct        10800 non-null  float64
 5   fitness_level            10800 non-null  object 
 6   experience_years         10800 non-null  int64  
 7   resting_heart_rate       9941 non-null   float64
 8   max_heart_rate           10800 non-null  int64  
 9   heart_rate_reserve       10800 non-null  int64  
 10  vo2_max_estimate         10800 non-null  float64
 11  fitness_goal             10800 non-null  object 
 12  time_available_min       10800 non-null  int64  
 13  weekly_consistency_days  10800 non-null  int64  
 14  workout_adherence_pct 

,row_id,timestamp,user_age,user_bmi,user_body_fat_pct,fitness_level,experience_years,resting_heart_rate,max_heart_rate,heart_rate_reserve,...,motivation_score,workout_name,muscle_groups,equipment_needed,target_workout_type,target_duration_min,target_intensity_level,estimated_calories,data_source,sleep_hours
count,10800.000000,10800,9181.000000,9463.000000,10800.000000,10800,10800.000000,9941.000000,10800.000000,10800.000000,...,9447.000000,10800,10800,10800,10761,9938.000000,10800.000000,9424.000000,10800,130.000000
unique,NaN,9101,NaN,NaN,NaN,15,NaN,NaN,NaN,NaN,...,NaN,1852,125,2,8,NaN,NaN,NaN,3,NaN
top,NaN,2025-06-12T00:16:17.098671,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,...,NaN,Restorative Flow,full_body,True,strength,NaN,NaN,NaN,bodybuilding,NaN
freq,NaN,6,NaN,NaN,NaN,5570,NaN,NaN,NaN,NaN,...,NaN,396,3475,9877,5392,NaN,NaN,NaN,5482,NaN
mean,5000.391944,NaN,30.030389,25.754031,21.739694,NaN,2.012500,63.112564,190.194630,126.808241,...,5.743728,NaN,NaN,NaN,NaN,35.957034,6.791296,83.298812,NaN,17.530769
std,2888.516795,NaN,11.163689,6.851481,6.016022,NaN,2.658466,9.048556,9.172034,12.015677,...,2.404266,NaN,NaN,NaN,NaN,53.337015,1.948964,45.788057,NaN,14.386788
min,1.000000,NaN,5.000000,2.000000,2.000000,NaN,0.000000,10.000000,145.000000,74.000000,...,1.000000,NaN,NaN,NaN,NaN,-15.000000,1.000000,11.000000,NaN,-3.000000
25%,2499.000000,NaN,22.000000,22.800000,17.600000,NaN,0.000000,58.000000,185.000000,119.000000,...,4.000000,NaN,NaN,NaN,NaN,20.000000,6.000000,50.000000,NaN,-3.000000
50%,5005.500000,NaN,29.000000,25.200000,21.700000,NaN,1.000000,63.000000,191.000000,128.000000,...,6.000000,NaN,NaN,NaN,NaN,31.000000,7.000000,73.000000,NaN,25.000000
75%,7509.250000,NaN,35.000000,27.800000,25.700000,NaN,3.000000,69.000000,197.000000,135.000000,...,8.000000,NaN,NaN,NaN,NaN,42.000000,9.000000,107.000000,NaN,30.000000


# 5. Copy Dataset

In [ ]:
df = df_raw.copy()

# 6. Drop Unnecessary Columns

In [ ]:
drop_columns = [
    'row_id',
    'timestamp',
    'data_source',
    'workout_name',
    'muscle_groups',
    'equipment_needed',
    'sleep_hours'
]

existing_drop_columns = [
    col for col in drop_columns
    if col in df.columns
]

df.drop(
    columns=existing_drop_columns,
    inplace=True
)

print(df.shape)

(10800, 27)


# 7. Clean Target Labels

In [ ]:
valid_classes = [
    'strength',
    'hiit',
    'flexibility'
]

def clean_target(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in valid_classes:
        return value

    if 'strength' in value:
        return 'strength'

    if 'hiit' in value:
        return 'hiit'

    if 'flex' in value:
        return 'flexibility'

    return np.nan

df['target_workout_type'] = (
    df['target_workout_type']
    .apply(clean_target)
)

df.dropna(
    subset=['target_workout_type'],
    inplace=True
)

print(df['target_workout_type'].value_counts())

target_workout_type
strength       5429
hiit           3407
flexibility    1801
Name: count, dtype: int64


# 8. Handle Missing Values

In [ ]:
numeric_cols = df.select_dtypes(
    include=['int64','float64']
).columns

categorical_cols = df.select_dtypes(
    include=['object']
).columns

num_imputer = SimpleImputer(
    strategy='median'
)

cat_imputer = SimpleImputer(
    strategy='most_frequent'
)

df[numeric_cols] = num_imputer.fit_transform(
    df[numeric_cols]
)

df[categorical_cols] = cat_imputer.fit_transform(
    df[categorical_cols]
)

print(df.isnull().sum().sum())

0


# 9. Remove Outliers

In [ ]:
outlier_columns = [
    'user_age',
    'user_bmi',
    'resting_heart_rate',
    'target_duration_min'
]

for col in outlier_columns:

    if col in df.columns:

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - (1.5 * iqr)
        upper = q3 + (1.5 * iqr)

        df = df[
            (df[col] >= lower) &
            (df[col] <= upper)
        ]

print(df.shape)

(9450, 27)


# 10. Feature Engineering

In [ ]:
# Convert all numeric columns to numeric, handling invalid values
numeric_cols_to_convert = [
    'fitness_level',
    'vo2_max_estimate',
    'resting_heart_rate',
    'heart_rate_reserve',
    'max_heart_rate',
    'experience_years'
]

for col in numeric_cols_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors='coerce'
        )
        # Fill any NaN values created from invalid conversions with the median
        df[col].fillna(
            df[col].median(),
            inplace=True
        )

df['cardio_capacity_index'] = (
    df['vo2_max_estimate']
    /
    (df['resting_heart_rate'] + 1)
)

df['heart_rate_efficiency'] = (
    df['heart_rate_reserve']
    /
    (df['max_heart_rate'] + 1)
)

df['fitness_experience_ratio'] = (
    df['fitness_level']
    /
    (df['experience_years'] + 1)
)

df.head()

TypeError: unsupported operand type(s) for /: 'str' and 'float'

# 11. Encode Target

In [ ]:
target_mapping = {
    'strength': 0,
    'hiit': 1,
    'flexibility': 2
}

df['target_encoded'] = (
    df['target_workout_type']
    .map(target_mapping)
)

print(df['target_encoded'].value_counts())

# 12. Feature Selection

In [ ]:
FEATURES = [

    'user_age',
    'user_bmi',
    'fitness_level',
    'experience_years',
    'resting_heart_rate',
    'heart_rate_reserve',
    'vo2_max_estimate',
    'motivation_score',
    'diet_quality_score',
    'hydration_liters',
    'workout_adherence_pct',
    'time_available_min',
    'weekly_consistency_days',
    'cardio_capacity_index',
    'heart_rate_efficiency',
    'fitness_experience_ratio'
]

X = df[FEATURES]

y = df['target_encoded']

print(X.shape)
print(y.shape)

# 13. Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

# 14. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

# 15. Define Models

In [ ]:
models = {

    'Logistic Regression':
        LogisticRegression(
            max_iter=1000,
            random_state=42
        ),

    'KNN':
        KNeighborsClassifier(
            n_neighbors=7
        ),

    'Random Forest':
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=42
        ),

    'Gradient Boosting':
        GradientBoostingClassifier(
            random_state=42
        ),

    'XGBoost':
        XGBClassifier(
            objective='multi:softprob',
            num_class=3,
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            eval_metric='mlogloss',
            random_state=42
        )
}

# 16. Train and Evaluate Models

In [ ]:
results = []

for name, model in models.items():

    print('=' * 60)
    print(name)
    print('=' * 60)

    if name in [
        'Logistic Regression',
        'KNN'
    ]:

        model.fit(
            X_train_scaled,
            y_train
        )

        preds = model.predict(
            X_test_scaled
        )

    else:

        model.fit(
            X_train,
            y_train
        )

        preds = model.predict(
            X_test
        )

    accuracy = accuracy_score(
        y_test,
        preds
    )

    precision = precision_score(
        y_test,
        preds,
        average='weighted'
    )

    recall = recall_score(
        y_test,
        preds,
        average='weighted'
    )

    f1 = f1_score(
        y_test,
        preds,
        average='weighted'
    )

    print(
        classification_report(
            y_test,
            preds
        )
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })

# 17. Results Table

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1',
    ascending=False
)

results_df

# 18. Best Model

In [ ]:
best_model_name = results_df.iloc[0]['Model']

print('Best Model:', best_model_name)

# 19. Final XGBoost Model

In [ ]:
final_model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=42
)

final_model.fit(
    X_train,
    y_train
)

final_preds = final_model.predict(
    X_test
)

print(
    classification_report(
        y_test,
        final_preds
    )
)

# 20. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    final_preds
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

# 21. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': final_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

importance_df

# 22. Feature Importance Plot

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=importance_df,
    x='Importance',
    y='Feature'
)

plt.title('Feature Importance')

plt.show()

# 23. Cross Validation

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    final_model,
    X,
    y,
    cv=cv,
    scoring='f1_weighted'
)

print(cv_scores)

print('Mean F1:', cv_scores.mean())

# 24. Statistical Significance Test

In [ ]:
baseline_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_train_scaled,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

xgb_scores = cross_val_score(
    final_model,
    X,
    y,
    cv=5,
    scoring='f1_weighted'
)

t_stat, p_value = stats.ttest_rel(
    xgb_scores,
    baseline_scores
)

print('P-Value:', p_value)

# 25. Final Findings

In [ ]:
print('Expected Findings:')
print('- XGBoost outperformed baseline models')
print('- Preprocessing improved dataset quality')
print('- Behavioral features strongly influence recommendations')
print('- Engineered features improved performance')
print('- Multi-class recommendation is more realistic academically')